# AI Similarity vs Fuzzy Matching: Reducing False Positives

**Problem:** Character-based fuzzy matching (RapidFuzz, Levenshtein) gives high scores when strings share common words (e.g. "holdings"), so pairs like **"fnz holdings"** vs **"panli holdings"**, **"ascent holdings"**, **"pepsico holdings"** all score ~80 and require manual review—even though they are unrelated companies.

**Approach:** Use Databricks **`ai_similarity()`** (semantic similarity) and optionally **`ai_query()`** to rank or classify pairs by *meaning*, so true matches stay high and false positives drop.

**Requirements:** Databricks Runtime 15.1+ in a region that supports [AI Functions](https://docs.databricks.com/aws/en/resources/feature-region-support#ai-aws).

## 1. Example records (RapidFuzz-style pairs)

These mirror the customer's data: same `match1`, several `match2` values that all get similar Levenshtein/RapidFuzz scores (~80) but are semantically different.

### Alternative: same data in pure SQL (for SQL notebooks / DBSQL)

If you prefer not to use Python, create the same example data with a CTE and run the `ai_similarity` query below.

In [ ]:
-- Pure SQL: example pairs (no Python). Run this then the next ai_similarity query.
WITH fuzzy_match_pairs(match1, match2, rapidfuzz_score) AS (
  SELECT 'fnz holdings', 'panli holdings', 80   UNION ALL
  SELECT 'fnz holdings', 'ascent holdings', 80   UNION ALL
  SELECT 'fnz holdings', 'pepsico holdings', 80  UNION ALL
  SELECT 'fnz holdings', 'fnz holdings ltd', 85  UNION ALL
  SELECT 'fnz holdings', 'fnz group', 72        UNION ALL
  SELECT 'pepsico holdings', 'pepsico inc', 78   UNION ALL
  SELECT 'pepsico holdings', 'coca cola holdings', 75
)
SELECT match1, match2, rapidfuzz_score, ai_similarity(match1, match2) AS ai_similarity_score
FROM fuzzy_match_pairs
ORDER BY ai_similarity_score DESC

In [ ]:
# Example pairs: match1 = reference, match2 = candidate (RapidFuzz would give ~80 for all "* holdings" pairs)
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("match1", StringType()),
    StructField("match2", StringType()),
    StructField("rapidfuzz_score", IntegerType()),  # simulated ~80 for partial matches
])

rows = [
    Row("fnz holdings", "panli holdings", 80),
    Row("fnz holdings", "ascent holdings", 80),
    Row("fnz holdings", "pepsico holdings", 80),
    # True positive: same entity, different wording
    Row("fnz holdings", "fnz holdings ltd", 85),
    Row("fnz holdings", "fnz group", 72),
    # Another reference to compare
    Row("pepsico holdings", "pepsico inc", 78),
    Row("pepsico holdings", "coca cola holdings", 75),
]

df = spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("fuzzy_match_pairs")
display(df)

## 2. Add semantic similarity with `ai_similarity()`

**`ai_similarity(expr1, expr2)`** returns a FLOAT (0–1, 1 = identical). It uses an embedding model, so "fnz holdings" vs "pepsico holdings" should score **lower** than "fnz holdings" vs "fnz holdings ltd". Use this to **rank** pairs and **filter** out low semantic scores to reduce manual review.

In [ ]:
%sql
-- Semantic similarity: unrelated "* holdings" pairs should score lower than same-entity pairs
SELECT
  match1,
  match2,
  rapidfuzz_score,
  ai_similarity(match1, match2) AS ai_similarity_score
FROM fuzzy_match_pairs
ORDER BY ai_similarity_score DESC

## 3. Weed out false positives with a threshold

Use a **semantic threshold** (e.g. ≥ 0.7) to keep likely same-entity pairs and send only those for review; drop rows where both RapidFuzz is high but `ai_similarity` is low.

In [ ]:
%sql
-- Example: require BOTH high RapidFuzz AND high ai_similarity to avoid false positives
WITH scored AS (
  SELECT
    match1,
    match2,
    rapidfuzz_score,
    ai_similarity(match1, match2) AS ai_similarity_score
  FROM fuzzy_match_pairs
)
SELECT
  *,
  CASE
    WHEN ai_similarity_score >= 0.7 THEN 'likely_same_entity'
    WHEN ai_similarity_score >= 0.5 THEN 'review'
    ELSE 'likely_false_positive'
  END AS recommendation
FROM scored
ORDER BY ai_similarity_score DESC

## 4. Optional: `ai_query()` for explicit same/different classification

For a small set of pairs or as a second pass, use **`ai_query()`** with a foundation model to ask "Are these the same company/entity?" and get a structured answer. Replace the endpoint name with your workspace's foundation model endpoint (e.g. `databricks-meta-llama-3-3-70b-instruct`).

In [ ]:
%sql
-- Optional: use ai_query to classify same entity vs different (replace endpoint with your foundation model)
-- Uncomment and run if you have Foundation Model APIs / endpoint available
/*
SELECT
  match1,
  match2,
  rapidfuzz_score,
  ai_similarity(match1, match2) AS ai_similarity_score,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'Are these two company or organization names referring to the SAME entity? Answer with only one word: SAME or DIFFERENT.\nName 1: ' || match1 || '\nName 2: ' || match2
  ) AS entity_verdict
FROM fuzzy_match_pairs
ORDER BY match1, match2;
*/

## Summary

| Method | Strength | Use in pipeline |
|--------|----------|------------------|
| **RapidFuzz / Levenshtein** | Fast, catches typos and small edits | First pass: keep pairs above score threshold |
| **`ai_similarity()`** | Semantic: same entity vs same word "holdings" | Second pass: rank/filter by semantic score to **weed out false positives** |
| **`ai_query()`** | Explicit yes/no "same entity?" | Optional: small sets or borderline cases for a final label |

**Suggested flow:** Keep RapidFuzz for speed; add `ai_similarity(match1, match2)` and only send pairs with **both** high RapidFuzz and high `ai_similarity` to manual review (or auto-approve when both are above chosen thresholds).